# Triangle RIG experiment: analyze saved measurements

This notebook reproduces the statistical summaries and transition plot
from the original `3_cores_in_random_intersection_graphs(1).ipynb`, using
the raw measurements saved by `03_triangle_simulation.ipynb`.

It runs independently in a fresh kernel: **no graph construction or
simulation is executed here**. Use the project's `.venv` kernel and edit
the **Analysis settings** cell below to select a saved CSV and figure
options. The final cell replaces the selected figure file and also
displays the figure inline.

In [ ]:
%matplotlib inline

import csv
from pathlib import Path

import numpy as np
from IPython.display import display
from matplotlib.figure import Figure

# Support kernels started in notebooks/ or at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Analysis settings

Select the CSV written by the simulation notebook and choose the figure
path and smoothing width here. Paths are relative to the project root.
The smoothing width must be a positive odd integer; it affects only the
finite-size slope indicator, not the plotted raw summaries.

Simulation parameters are recorded in the CSV, and plot titles use those
recorded values. Changing analysis settings never runs a new simulation.
To measure a new graph size or lambda grid, edit and rerun notebook 03.

In [ ]:
config = {
    "data_file": "results/runs/triangle_transition.csv",  # Match notebook 03's output.
    "figure_file": "results/figures/triangle_transition.png",  # Replaced when plotted.
    "smoothing_window": 7,  # Positive odd width for the slope indicator only.
}
config

## Read and check the raw measurements

The CSV records individual realizations, rather than only averages. This
makes the spread near the transition visible and allows seeds to be
replayed in the simulation notebook. Integer seeds are read directly as
integers; reading them through floating point could lose precision.
The loader checks the schema and verifies that each fraction agrees with
its saved core count and graph size.

In [ ]:
def load_results(path: str | Path) -> list[dict]:
    """Read typed CSV rows and check their core count/fraction agreement."""
    fields = ["model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction"]
    rows = []
    with Path(path).open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != fields:
            raise ValueError(f"Expected CSV columns: {', '.join(fields)}")
        for row in reader:
            for key in ("n", "replicate", "seed", "core_size"):
                row[key] = int(row[key])
            row["lambda"] = float(row["lambda"])
            row["core_fraction"] = float(row["core_fraction"])
            if (
                row["model"] != "triangle_rig"
                or row["n"] <= 0
                or not 0 <= row["core_size"] <= row["n"]
                or not np.isfinite(row["lambda"])
                or row["lambda"] < 0
                or row["replicate"] < 0
                or row["seed"] < 0
                or not np.isclose(
                    row["core_fraction"],
                    row["core_size"] / row["n"],
                    rtol=1e-9,
                    atol=1e-12,
                )
            ):
                raise ValueError("Invalid measurement or inconsistent core fraction")
            rows.append(row)
    if not rows:
        raise ValueError("The CSV contains no measurements")
    return rows

## Summary statistics and a finite-size slope indicator

For each lambda, compute the mean and median core fraction, the minimum
and maximum, and the 25th and 75th percentiles. The two shaded bands show
**variation between sampled graphs**; they are not confidence intervals
for the mean. Large differences between mean and median can help reveal
a mixture of small-core and large-core realizations near the transition.

As in the original experiment, smooth the mean with a moving average,
estimate its derivative on the lambda grid, and locate the largest
**interior** positive slope. Edge padding repeats the endpoint means.
The smoothing window is reduced to fit a short grid when necessary.
Fewer than three grid points, flat means, or no positive interior slope
produce no estimate.

This is a descriptive **finite-size heuristic**, not a theoretical
critical parameter. Its location depends on $n$, the grid, the number of
realizations, and the smoothing window. Compare several graph sizes and
eventually the cavity prediction before drawing asymptotic conclusions.

In [ ]:
def estimate_transition(lambdas, means, window: int = 7) -> float | None:
    """Return the lambda of the largest interior smoothed positive slope."""
    if window < 1 or window % 2 == 0:
        raise ValueError("The smoothing window must be a positive odd integer")
    if len(lambdas) < 3 or np.allclose(
        means, means[0], rtol=0, atol=np.finfo(float).eps
    ):
        return None

    # An odd window preserves one smoothed value per input grid point.
    window = min(window, len(lambdas) if len(lambdas) % 2 else len(lambdas) - 1)
    padded = np.pad(means, window // 2, mode="edge")
    smoothed = np.convolve(padded, np.ones(window) / window, mode="valid")
    slopes = np.gradient(smoothed, lambdas)
    index = int(np.argmax(slopes[1:-1])) + 1
    return float(lambdas[index]) if slopes[index] > 0 else None


def plot_results(rows: list[dict], smoothing_window: int = 7):
    """Return a figure and slope estimate for one graph size.

    Rows may be in any order. Bands summarize realization spread at
    each lambda, while mean and median are computed from the raw trials.
    """
    if not rows or len({row["n"] for row in rows}) != 1:
        raise ValueError("Provide measurements for exactly one graph size")
    grouped = {}
    for row in rows:
        grouped.setdefault(row["lambda"], []).append(row["core_fraction"])
    lambdas = np.array(sorted(grouped))
    samples = [grouped[lam] for lam in lambdas]
    means = np.array([np.mean(sample) for sample in samples])
    medians = [np.median(sample) for sample in samples]
    minima = [min(sample) for sample in samples]
    maxima = [max(sample) for sample in samples]
    q25 = [np.percentile(sample, 25) for sample in samples]
    q75 = [np.percentile(sample, 75) for sample in samples]
    estimate = estimate_transition(lambdas, means, smoothing_window)

    figure = Figure(figsize=(10, 6), layout="constrained")
    axis = figure.subplots()
    axis.fill_between(
        lambdas,
        minima,
        maxima,
        color="tab:blue",
        alpha=0.12,
        label="Min–max range",
    )
    axis.fill_between(
        lambdas,
        q25,
        q75,
        color="tab:blue",
        alpha=0.3,
        label="25th–75th percentile",
    )
    axis.plot(lambdas, means, color="tab:blue", linewidth=2, label="Mean")
    axis.plot(
        lambdas,
        medians,
        color="tab:orange",
        linestyle="--",
        linewidth=1.5,
        label="Median",
    )
    if estimate is not None:
        axis.axvline(
            estimate,
            color="tab:red",
            linestyle="--",
            linewidth=2,
            label=f"Finite-size slope estimate: {estimate:.3f}",
        )
    counts = [len(sample) for sample in samples]
    repeats = (
        str(counts[0]) if min(counts) == max(counts) else f"{min(counts)}–{max(counts)}"
    )
    axis.set_title(
        f"3-core transition in triangle RIGs (n={rows[0]['n']}, {repeats} trials per λ)"
    )
    axis.set_xlabel(r"$\lambda$")
    axis.set_ylabel(r"3-core fraction $|\mathrm{Core}_3(G)|/n$")
    if len(lambdas) > 1:
        axis.set_xlim(lambdas[0], lambdas[-1])
    axis.set_ylim(0, 1)
    axis.grid(True, alpha=0.3)
    axis.legend()
    return figure, estimate

In [ ]:
data_path = PROJECT_ROOT / config["data_file"]
if not data_path.is_file():
    raise FileNotFoundError(
        f"No saved measurements at {data_path}. "
        "Run make simulate from the project root, or run notebook 03 first."
    )
rows = load_results(data_path)
print(
    f"Loaded {len(rows)} realizations at "
    f"{len({row['lambda'] for row in rows})} lambda values from {data_path}."
)

## Plot, save, and interpret

The horizontal axis is the mean membership parameter $\lambda$, and the
vertical axis is $|\operatorname{Core}_3(G)|/n$. Look at both the central
tendency and the spread: averaging across realizations can obscure a
mixture of outcomes. A transition indicator close to the grid boundary
suggests widening the grid before interpreting its location.

In [ ]:
figure, transition_estimate = plot_results(rows, config["smoothing_window"])
figure_path = PROJECT_ROOT / config["figure_file"]
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=160)
display(figure)

print(f"Saved figure to {figure_path}.")
if transition_estimate is None:
    print("No finite-size slope estimate for this short or flat curve.")
else:
    print(f"Finite-size maximum-slope location: {transition_estimate:.4f}")